# Full Testmini Reevaluation: Baseline and Smoke Checkpoints

Reevaluate the local pre-refactor baseline adapter plus the smoke Phase B and smoke Phase C best checkpoints on the full `testmini` split.

This notebook is separate from the original training notebooks. It does not overwrite any prior
Kaggle run outputs. It expects:

- the reevaluation code dataset attachment,
- adapter/checkpoint attachments for the targets listed in `TARGET_SPECS`, and
- enough GPU time to run a full `testmini` reevaluation pass.

If target discovery is ambiguous or missing, edit the `TARGET_SPECS` cell and rerun.


In [ ]:
import glob, json, os, shutil, subprocess, tarfile
from datetime import datetime

WORKDIR = "/kaggle/working/RL_GSPO_Qwen2.5VLM"
VENV_DIR = "/tmp/rl-gspo-venv"
VENV_PYTHON = f"{VENV_DIR}/bin/python"
PROGRESS_PATH = "/kaggle/working/progress.txt"
os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"


def write_progress(message):
    timestamp = datetime.utcnow().isoformat()
    line = f"{timestamp} {message}"
    with open(PROGRESS_PATH, "a", encoding="utf-8") as handle:
        handle.write(line + "\n")
    print(line)

write_progress("Notebook start")


def find_code_dataset_root():
    matches = []
    for root, _, files in os.walk("/kaggle/input"):
        if "reeval_bundle_manifest.json" in files and "rl_gspo_qwen2_5vlm_eval.py" in files:
            matches.append(root)
    if not matches:
        raise FileNotFoundError(
            f"Could not find attached code dataset. Visible inputs: {glob.glob('/kaggle/input/*')}"
        )
    if len(matches) > 1:
        raise RuntimeError(
            f"Ambiguous code dataset attachment. Matches={matches}. Visible inputs: {glob.glob('/kaggle/input/*')}"
        )
    return matches[0]


def load_json_if_exists(path):
    if not os.path.exists(path):
        return {}
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def catalog_attached_adapters():
    candidates = []
    for root, _, files in os.walk("/kaggle/input"):
        if "adapter_model.safetensors" not in files:
            continue
        checkpoint_info_path = os.path.join(root, "checkpoint_info.json")
        reward_weights_path = os.path.join(root, "reward_weights.json")
        checkpoint_info = load_json_if_exists(checkpoint_info_path)
        candidate = {
            "root": root,
            "checkpoint_name": os.path.basename(root),
            "phase_name": checkpoint_info.get("phase_name"),
            "source_phase_name": checkpoint_info.get("source_phase_name"),
            "checkpoint_path_from_info": checkpoint_info.get("checkpoint_path"),
            "reward_weights_path": reward_weights_path if os.path.exists(reward_weights_path) else None,
            "checkpoint_info_path": checkpoint_info_path if os.path.exists(checkpoint_info_path) else None,
            "metrics": checkpoint_info.get("metrics", {}),
        }
        haystack_parts = [
            candidate["root"],
            candidate["checkpoint_name"],
            candidate["phase_name"],
            candidate["source_phase_name"],
            candidate["checkpoint_path_from_info"],
        ]
        candidate["haystack"] = " ".join(str(part).lower() for part in haystack_parts if part)
        candidates.append(candidate)
    return sorted(candidates, key=lambda item: item["root"])


CODE_DATASET_ROOT = find_code_dataset_root()
CATALOG = catalog_attached_adapters()

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
shutil.copytree(CODE_DATASET_ROOT, WORKDIR)

for archive_name in ("staged_rl.tar",):
    archive_path = os.path.join(WORKDIR, archive_name)
    if os.path.exists(archive_path):
        with tarfile.open(archive_path, "r") as tf:
            tf.extractall(WORKDIR)
        os.remove(archive_path)
        print("Extracted", archive_name)

for folder_name in ("staged_rl",):
    nested_path = os.path.join(WORKDIR, folder_name, folder_name)
    target_path = os.path.join(WORKDIR, folder_name)
    if os.path.isdir(nested_path):
        temp_path = f"{target_path}_flat"
        if os.path.exists(temp_path):
            shutil.rmtree(temp_path)
        shutil.move(nested_path, temp_path)
        shutil.rmtree(target_path)
        shutil.move(temp_path, target_path)
        print("Flattened", folder_name)

print("Using code dataset", CODE_DATASET_ROOT)
print("Copied code to", WORKDIR)
print("Visible input roots:", glob.glob("/kaggle/input/*"))
print("Discovered adapter candidates:")
print(json.dumps(CATALOG, indent=2))
write_progress("Copied code dataset and cataloged adapters")


In [ ]:
write_progress("Installing uv and creating venv")
subprocess.run(["python3", "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["python3", "-m", "uv", "venv", "--seed", "--clear", VENV_DIR], check=True)
install_commands = [
    [VENV_PYTHON, "-m", "pip", "install", "--no-cache-dir", "--upgrade", "pip", "setuptools", "wheel"],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "numpy==1.26.4",
        "scipy==1.15.3",
        "scikit-learn==1.6.1",
    ],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "pillow",
        "packaging",
        "safetensors",
        "torchvision",
        "bitsandbytes",
        "xformers",
        "huggingface_hub>=0.34.0",
        "datasets==4.3.0",
        "transformers==4.56.2",
        "trl==0.22.2",
        "unsloth",
        "modelscope",
    ],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "vllm==0.10.2",
        "--extra-index-url",
        "https://wheels.vllm.ai/0.10.2/",
    ],
]
for command in install_commands:
    print("Running:", " ".join(command))
    subprocess.run(command, check=True, cwd=WORKDIR)
write_progress("Finished venv package installs")
print("Venv ready:", VENV_PYTHON)


In [ ]:
compat_script = r'''
import os
os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"
import numpy
import scipy
import sklearn
import torch
import transformers
import trl
import unsloth
import vllm
print({
    "numpy": numpy.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "trl": trl.__version__,
    "vllm": vllm.__version__,
})
'''
subprocess.run([VENV_PYTHON, "-c", compat_script], check=True, cwd=WORKDIR)
write_progress("Verified dependencies in venv")


In [ ]:
baseline_script = r'''
import json
import os
import shutil
import torch
from safetensors.torch import load_file, save_file


def load_json(path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def find_rank8_adapter():
    candidates = []
    for root, _, files in os.walk("/kaggle/input"):
        if "adapter_config.json" not in files or "adapter_model.safetensors" not in files:
            continue
        try:
            cfg = load_json(os.path.join(root, "adapter_config.json"))
        except Exception:
            continue
        try:
            rank = int(cfg.get("r", 0))
        except Exception:
            continue
        if rank == 8:
            candidates.append(root)
    if not candidates:
        raise RuntimeError("No rank-8 adapter found under /kaggle/input.")
    for root in candidates:
        if "phase_a" in root:
            return root
    return sorted(candidates)[0]


template_root = find_rank8_adapter()
baseline_root = os.path.join(os.environ["WORKDIR"], "baseline_rank8")
if os.path.exists(baseline_root):
    shutil.rmtree(baseline_root)
os.makedirs(baseline_root, exist_ok=True)
shutil.copy(os.path.join(template_root, "adapter_config.json"), baseline_root)

tensors = load_file(os.path.join(template_root, "adapter_model.safetensors"))
zeroed = {key: torch.zeros_like(value) for key, value in tensors.items()}
save_file(zeroed, os.path.join(baseline_root, "adapter_model.safetensors"))

with open(os.path.join(baseline_root, "checkpoint_info.json"), "w", encoding="utf-8") as handle:
    json.dump(
        {
            "phase_name": "baseline",
            "checkpoint_name": "baseline_rank8",
            "checkpoint_path": "baseline_rank8",
            "source_phase_name": "baseline",
            "notes": "Rank-8 zeroed LoRA adapter for base-model evaluation.",
        },
        handle,
        indent=2,
    )
print("Baseline rank8 adapter:", baseline_root)
print("Template adapter:", template_root)
'''
env = dict(os.environ)
env["WORKDIR"] = WORKDIR
subprocess.run([VENV_PYTHON, "-c", baseline_script], check=True, env=env)
BASELINE_RANK8_DIR = os.path.join(WORKDIR, "baseline_rank8")
print("Baseline rank8 dir:", BASELINE_RANK8_DIR)
write_progress("Prepared baseline rank8 adapter")


In [ ]:
HARDWARE_PROFILE = "kaggle_t4"
EVAL_SPLIT = "testmini"
OUTPUT_ROOT = "outputs_full_testmini_reeval_baseline_rank8_phasewise_4"
FULL_SPLIT = False
MAX_EVAL_EXAMPLES_PER_SUBSET = 4

TARGET_SPECS = [
  {
    "label": "baseline_rank8",
    "phase": "phase_a",
    "max_completion_length": 64,
    "checkpoint": BASELINE_RANK8_DIR
  }
]

def resolve_target(spec, catalog):
    if "checkpoint" in spec:
        resolved = {
            "label": spec["label"],
            "checkpoint": spec["checkpoint"],
            "phase": spec["phase"],
            "max_completion_length": spec["max_completion_length"],
        }
        if spec.get("reward_weights_json"):
            resolved["reward_weights_json"] = spec["reward_weights_json"]
        return resolved

    matches = []
    for candidate in catalog:
        haystack = candidate["haystack"]
        if any(token.lower() not in haystack for token in spec.get("match_all", [])):
            continue
        match_any = spec.get("match_any", [])
        if match_any and not any(token.lower() in haystack for token in match_any):
            continue
        matches.append(candidate)

    if not matches:
        raise RuntimeError(
            "Could not resolve target "
            + spec["label"]
            + ". Edit TARGET_SPECS or attach the missing dataset.\nCatalog="
            + json.dumps(catalog, indent=2)
        )
    if len(matches) > 1:
        raise RuntimeError(
            "Target resolution is ambiguous for "
            + spec["label"]
            + ". Tighten the match tokens.\nMatches="
            + json.dumps(matches, indent=2)
        )

    match = matches[0]
    resolved = {
        "label": spec["label"],
        "checkpoint": match["root"],
        "phase": spec["phase"],
        "max_completion_length": spec["max_completion_length"],
    }
    if match.get("reward_weights_path"):
        resolved["reward_weights_json"] = match["reward_weights_path"]
    return resolved

RESOLVED_TARGETS = [resolve_target(spec, CATALOG) for spec in TARGET_SPECS]
TARGET_SPEC_PATH = os.path.join(WORKDIR, f"{OUTPUT_ROOT}_targets.json")
with open(TARGET_SPEC_PATH, "w", encoding="utf-8") as handle:
    json.dump({"targets": RESOLVED_TARGETS}, handle, indent=2)

print("Resolved targets:")
print(json.dumps(RESOLVED_TARGETS, indent=2))
print("Target spec path:", TARGET_SPEC_PATH)
write_progress("Resolved target spec")


In [ ]:
cmd = [
    VENV_PYTHON,
    "rl_gspo_qwen2_5vlm_eval.py",
    "--target-spec-json", TARGET_SPEC_PATH,
    "--hardware-profile", HARDWARE_PROFILE,
    "--output-root", OUTPUT_ROOT,
    "--eval-split", EVAL_SPLIT,
]
if MAX_EVAL_EXAMPLES_PER_SUBSET is not None:
    cmd.extend(["--max-eval-examples-per-subset", str(MAX_EVAL_EXAMPLES_PER_SUBSET)])
if FULL_SPLIT:
    cmd.append("--full-split")

env = dict(os.environ)
env["PYTHONUNBUFFERED"] = "1"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["UNSLOTH_USE_MODELSCOPE"] = "1"

reeval_log_path = os.path.join(WORKDIR, OUTPUT_ROOT, "reevaluation_subprocess.log")
os.makedirs(os.path.dirname(reeval_log_path), exist_ok=True)
print("Running:", " ".join(cmd))
print("Subprocess log:", reeval_log_path)
write_progress("Starting reevaluation subprocess")
with open(reeval_log_path, "w", encoding="utf-8") as log_file:
    completed = subprocess.run(
        cmd,
        check=False,
        cwd=WORKDIR,
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
if completed.returncode != 0:
    write_progress("Reevaluation subprocess failed")
    print(f"Reevaluation failed with return code {completed.returncode}. Last 200 log lines:")
    with open(reeval_log_path, "r", encoding="utf-8") as handle:
        lines = handle.readlines()[-200:]
    print("".join(lines))
    raise RuntimeError(f"Reevaluation subprocess failed with return code {completed.returncode}")
write_progress("Reevaluation finished successfully")
print("Reevaluation finished successfully.")


In [ ]:
summary_path = os.path.join(WORKDIR, OUTPUT_ROOT, "reevaluation_summary.json")
csv_path = os.path.join(WORKDIR, OUTPUT_ROOT, "reevaluation_summary.csv")
subset_sizes_path = os.path.join(WORKDIR, OUTPUT_ROOT, "eval_subset_sizes.json")

with open(subset_sizes_path, "r", encoding="utf-8") as handle:
    print("Subset sizes:")
    print(handle.read())

with open(summary_path, "r", encoding="utf-8") as handle:
    print("JSON summary:")
    print(handle.read())

print("CSV summary path:", csv_path)

collected = []
for root, _, files in os.walk(os.path.join(WORKDIR, OUTPUT_ROOT)):
    for file_name in files:
        collected.append(os.path.join(root, file_name))
for path in sorted(collected):
    print(path)
progress_copy = os.path.join(WORKDIR, OUTPUT_ROOT, "progress.txt")
if os.path.exists(PROGRESS_PATH):
    shutil.copy(PROGRESS_PATH, progress_copy)
    print("Progress log copied to", progress_copy)
write_progress("Summary artifacts listed")
